In [1]:
import torch
import lightning
from minicons import cwe
import pandas as pd
import os
import glob
import re
import numpy as np
from tqdm import tqdm

from model import FFNModule, FeatureNormPredictor, FFNParams, TrainingParams



/opt/conda/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/home/gsc685/.local/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/gsc685/.local/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [4]:

model = FeatureNormPredictor.load_from_checkpoint(
    checkpoint_path='/home/shared/semantic_features/saved_models/bert_models_all/bert_to_buchanan_layer7.ckpt',
    map_location=None
)
model = model.to('cuda') 

print("model hyperparameters: ")
for key,value in model.hparams.items():
    print("    {}: {}".format(key, value))

# get a sample embedding to test
data = [
    ("colorless green ideas sleep furiously", "ideas")
]
lm = cwe.CWE('bert-base-uncased')
emb = lm.extract_representation(data, layer=11)
predicted= model(emb.cuda())
squeezed = predicted.squeeze(0)
print(squeezed.shape)

ratings_df = pd.read_csv('feature-norms/binder/WordSet1_Ratings.csv', na_values=['na'])
# fill in 0 for na's
ratings_df.fillna(value=0, inplace=True)
feature_cols = ratings_df.iloc[:,5:70].columns



for i in range(len(feature_cols)):
    print(feature_cols[i]," : ", squeezed[i].item())

    


model hyperparameters: 
    ffn_params: input_size=768 output_size=3981 hidden_size=1344 num_layers=2 dropout=0.5
    training_params: num_epochs=100 batch_size=75 learning_rate=0.0007541899688915023 weight_decay=0.0


/home/gsc685/.local/lib/python3.9/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


torch.Size([3981])
65
3981
Vision  :  0.014619015157222748
Bright  :  0.1592768132686615
Dark  :  0.2016913890838623
Color  :  -0.08194465935230255
Pattern  :  0.44093745946884155
Large  :  0.06255216151475906
Small  :  1.3548275232315063
Motion  :  -0.04880519211292267
Biomotion  :  0.05655784159898758
Fast  :  -0.05896990746259689
Slow  :  0.0066246334463357925
Shape  :  0.03966894373297691
Complexity  :  -0.11503475159406662
Face  :  0.4339008629322052
Body  :  0.011071744374930859
Touch  :  0.35650694370269775
Temperature  :  0.032541632652282715
Texture  :  -0.06552807241678238
Weight  :  0.1997920274734497
Pain  :  -0.0009720958769321442
Audition  :  -0.513247013092041
Loud  :  -0.4573350250720978
Low  :  0.09455960243940353
High  :  0.00739723164588213
Sound  :  0.6549981832504272
Music  :  -0.12249200791120529
Speech  :  -0.20210453867912292
Taste  :  -0.14520660042762756
Smell  :  0.13185986876487732
Head  :  -0.0600496307015419
UpperLimb  :  0.4320376515388489
LowerLimb  :  -

In [ ]:
# get buchanan features

In [ ]:

model = FeatureNormPredictor.load_from_checkpoint(
    checkpoint_path='/home/shared/semantic_features/saved_models/bert_models_all/bert_to_buchanan_layer7.ckpt',
    map_location=None
)
model = model.to('cuda') 

print("model hyperparameters: ")
for key,value in model.hparams.items():
    print("    {}: {}".format(key, value))

# get a sample embedding to test
data = [
    ("colorless green ideas sleep furiously", "ideas")
]
lm = cwe.CWE('bert-base-uncased')
emb = lm.extract_representation(data, layer=11)
predicted= model(emb.cuda())
squeezed = predicted.squeeze(0)
print(squeezed.shape)

# buchanan
ratings_df = pd.read_csv('feature-norms/buchanan/cue_feature_words.csv', na_values=['na'])
# fill in 0 for na's
ratings_df.fillna(value=0, inplace=True)
feature_cols = ratings_df["translated"].unique()
print(feature_cols)


print(len(feature_cols))
print(len(squeezed))



for i in range(len(feature_cols)):
    print(feature_cols[i]," : ", squeezed[i].item())

In [ ]:
"""
predict features for all the tokens in the coca /acl corpus
"""

In [20]:


corpora = ["acl", "coca"]
data_dir = "/home/gsc685/data/collected_tokens/"
embedding_model = 'bert-base-uncased'
model_dir = '/home/shared/semantic_features/saved_models/bert_models_all/bert_to_buchanan_layer7.ckpt'
layer = 7


# helper function to batch process inputs
def batch_iterable(iterable, batch_size):
    for i in range(0, len(iterable), batch_size):
        yield iterable[i:i + batch_size]



# iterate through the token files
for corpus in corpora:
    
    token_files = glob.glob(data_dir + corpus + "/*.csv") 
    
    # just focus on one word for now
    token_files = [os.path.join(data_dir, corpus, 'model.csv')]
    for filename in token_files:
        
        
        """
        prepare the data
        """

        # pull the word we are predicting from the filename
        pattern = r'([a-zA-Z0-9_-]+)(?=\.csv$)'
        word = re.search(pattern, filename).group(0)
        print(word)
        
        tokens_path = os.path.join(data_dir, corpus, filename)
        tokens = pd.read_csv(tokens_path).sample(n=1000)
        tokens['word'] = word
        print(tokens.shape)
                
        # data as list of tuples
        data = list(zip(tokens['sentence'], tokens['word']))
        #print(data)
        
        """
        load the models 
        """
        lm = cwe.CWE(embedding_model)


        model = FeatureNormPredictor.load_from_checkpoint(
            checkpoint_path=model_dir,
            map_location=None
        ).to('cuda')

        print("model hyperparameters: ")
        for key,value in model.hparams.items():
            print("    {}: {}".format(key, value))
            
            
        # get length of output
        dummy = [ ("colorless green ideas sleep furiously", "ideas")]
        emb = lm.extract_representation(dummy, layer=layer)
        predicted= model(emb.cuda())
        squeezed = predicted.squeeze(0).cpu().detach().numpy()
        num_dims = squeezed.shape[0]
        print(num_dims)
        
        
        """
        run the model over the data
        """
    
        feats = [] * len(data)
        batch_size = 75
        #allfeats = np.empty((0, num_dims)).to('cuda') # zero rows and output-size columns
        #allfeats = torch.empty((0, num_dims), device='cuda')
        for i, batch in tqdm(enumerate(batch_iterable(data, batch_size))):
                       
            try:
                emb = lm.extract_representation(batch, layer=layer)
                predicted= model(emb.cuda())
                vecs = predicted.squeeze(0)
                
            except:
                vecs = torch.empty((batch_size,num_dims), device='cuda')
            #feats[i*batch_size:i*batch_size+batch_size] = vecs
            feats.append(vecs)
        
        feats = torch.cat(feats).detach().cpu().numpy()
        # buchanan
        # ratings_df = pd.read_csv('feature-norms/buchanan/cue_feature_words.csv', na_values=['na'])
        # # fill in 0 for na's
        # ratings_df.fillna(value=0, inplace=True)
        # feature_cols = ratings_df["translated"].unique()
        # print(feature_cols)
        
        
        outpath = os.path.join(data_dir, corpus, word + "_feature_vectors_bert_buchanan_layer7.txt")
        np.savetxt(outpath, feats)  # %d is used for integer formatting

        # for i in range(len(feature_cols)):
        #     print(feature_cols[i]," : ", squeezed[i].item())

model
(1000, 6)


/home/gsc685/.local/lib/python3.9/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model hyperparameters: 
    ffn_params: input_size=768 output_size=3981 hidden_size=1344 num_layers=2 dropout=0.5
    training_params: num_epochs=100 batch_size=75 learning_rate=0.0007541899688915023 weight_decay=0.0
3981


14it [00:22,  1.59s/it]


model
(1000, 6)


/home/gsc685/.local/lib/python3.9/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model hyperparameters: 
    ffn_params: input_size=768 output_size=3981 hidden_size=1344 num_layers=2 dropout=0.5
    training_params: num_epochs=100 batch_size=75 learning_rate=0.0007541899688915023 weight_decay=0.0
3981


14it [00:11,  1.20it/s]


In [21]:
torch.cuda.is_available()

True